In [66]:
# Install the project dependencies.
%pip install -q -r ../../requirements.txt


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

RUN = ROOT / "runs/estimation/archive/2026-06-03_10-22-50_income_IC_country_trends"
DATA = ROOT / "data/archive/2026-08-27-MainData-PREBUGFIX.xlsx"
FIGURES = ROOT / "paper/Figures/Income"

print("run    ", RUN.name)
print("exists ", RUN.exists())

In [ ]:
import base64
import json
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml

from utils import create_pred_input
from hdf5_min import read_weights
import paper_surface as ps

cfg = yaml.safe_load((RUN / "config/config.yaml").read_text(encoding="utf-8"))["instance"]

for k in ("formulation", "target_mode", "within_projection", "country_trends", "data_source"):
    print(f"{k:20} {cfg.get(k)}")

In [ ]:
data = pd.read_excel(DATA)

mean_temp = np.nanmean(data["TempPopWeight"])
std_temp = np.nanstd(data["TempPopWeight"])
mean_precip = np.nanmean(data["PrecipPopWeight"])
std_precip = np.nanstd(data["PrecipPopWeight"])

pred_input, T, P = create_pred_input(
    mc=False, mean_T=mean_temp, std_T=std_temp,
    mean_P=mean_precip, std_P=std_precip, precip_capped=True)

gdp = data[["CountryCode", "Year", "GDPCap"]].dropna(subset=["GDPCap"]).copy()
gdp["IncomeGroup"] = pd.qcut(gdp["GDPCap"], q=2, labels=["Low", "High"])
key = gdp.set_index(["CountryCode", "Year"])["IncomeGroup"]
income_grp = pd.Series(
    key.reindex(list(zip(data["CountryCode"], data["Year"]))).to_numpy(),
    index=data.index)

print("grid", T.shape, " mean T %.2f  mean P %.0f mm" % (mean_temp, mean_precip))

In [ ]:
results = dict(np.load(RUN / "results.npy", allow_pickle=True).item())
results = {k: v for k, v in results.items() if v is not None}

n_models = 10
top_models = sorted(results, key=lambda node: results[node][0])[:n_models]

for rank, node in enumerate(top_models, 1):
    print(f"{rank:>2}. {str(node):<14} BIC {float(results[node][0]):>12.1f}  AIC {float(results[node][1]):>12.1f}")

In [ ]:

############################## User Input ##############################
CENTRE_AT_SAMPLE_MEAN = False  # shift each surface to zero at mean (T, P)
GROUP = "High"                 # income group to plot, "Low" or "High"
########################################################################

HEADS = {"Low": 0, "High": 1}  # output-head order from prepare_data.py
HEAD = HEADS[GROUP]

def swish(z):
    return z / (1.0 + np.exp(-np.clip(z, -60.0, 60.0)))

def forward(w, X):
    order = lambda n: 0 if "_" not in n else int(n.split("_")[1])
    names = sorted({k.split("/")[1] for k in w if k.startswith("layers/")}, key=order)
    hidden = [n for n in names if f"layers/{n}/vars/1" in w]
    rest = [n for n in names if f"layers/{n}/vars/1" not in w]
    block = len(rest) // len(HEADS)
    h = X
    for k in hidden[HEAD::len(HEADS)]:
        h = swish(h @ np.asarray(w[f"layers/{k}/vars/0"], float)
                  + np.asarray(w[f"layers/{k}/vars/1"], float))
    head = next(n for n in rest[HEAD * block:(HEAD + 1) * block]
                if np.shape(w[f"layers/{n}/vars/0"]) == (h.shape[-1], 1))
    return (h @ np.asarray(w[f"layers/{head}/vars/0"], float))[..., 0]

ORIGIN = np.zeros((1,) * (pred_input.ndim - 1) + (pred_input.shape[-1],))

def surface(node):
    w = read_weights(str(RUN / "parameters" / f"{node}.weights.h5"))
    Z = forward(w, pred_input).reshape(T.shape)
    return Z - float(np.ravel(forward(w, ORIGIN))[0]) if CENTRE_AT_SAMPLE_MEAN else Z

surfaces = {node: surface(node) for node in top_models
            if (RUN / "parameters" / f"{node}.weights.h5").exists()}
z = np.mean(list(surfaces.values()), axis=0)

model_surfaces = [
    go.Surface(x=T, y=P / 1000, z=Z, colorscale='Cividis',
               opacity=0.85, showscale=False, name=f'Model {node}')
    for node, Z in surfaces.items()
]


In [ ]:
def published_surface(html_path):
    """The z grid of the surface trace in a saved plotly HTML export."""
    h = Path(html_path).read_text(encoding="utf-8", errors="ignore")
    call = None
    for call in re.finditer(r'Plotly\.newPlot\(\s*"[0-9a-f-]{36}"\s*,\s*', h):
        pass
    start = h.index("[", call.end() - 1)
    depth, i = 0, start
    while i < len(h):
        if h[i] in "[{":
            depth += 1
        elif h[i] in "]}":
            depth -= 1
        if depth == 0:
            break
        i += 1
    for trace in json.loads(h[start:i + 1]):
        if trace.get("type") == "surface":
            v = trace["z"]
            a = np.frombuffer(base64.b64decode(v["bdata"]), dtype=np.dtype(v["dtype"]))
            return a.reshape([int(s) for s in str(v["shape"]).split(",")]).astype(float)
    raise ValueError(f"no surface trace in {html_path}")


ref = ROOT / f"results/images/robustness/{GROUP}_CT.html"
assert np.max(np.abs(z - published_surface(ref))) < 1e-6, GROUP

In [ ]:
fig = go.Figure(data=model_surfaces[0:5])
fig.update_layout(
        autosize=True,

        margin=dict(
            l=0,
            r=0,
            b=0,
            t=0,
        ),

            scene=dict(
                xaxis_title='Temperature (°C)',
                yaxis_title='Precipitation (m)',
                zaxis=dict(title=dict(text="Δ ln(GDP)"),range=[-0.3, 0.3]),
                camera=dict(eye=dict(x=1.738, y=-1.780, z=0.589))

            ),

            legend=dict(
                bgcolor='rgba(255,255,255,0.7)',
                bordercolor='black',
                borderwidth=1
            ),
            font=dict(
            size=10
        )
        )
fig=ps.add_density_floor(fig, data, -0.3, income=GROUP, income_grp=income_grp, precip_capped=True)
fig.show()

In [ ]:
print(ps.write(fig, FIGURES / f"{RUN.name}_{GROUP}_CT", pdf=True, crop=True))